# MyTravels Infrastructure Runbook

This notebook is the end-to-end operational runbook for standing up the full MyTravels Kubernetes infrastructure on a local k3d cluster. Run cells top to bottom to bring everything up from scratch.

## Summary

- **Step 1 — Prerequisites**: Verify the required tools are installed (Rancher Desktop, k3d, kubectl, JupyterLab, OpenLens).
- **Step 2 — Create the Cluster**: Create the local k3d cluster (2 control plane, 4 workers) with port mappings for Traefik HTTP (8080) and PostgreSQL TCP (5432).
- **Step 3 — /etc/hosts**: Add the nine `*.mytravels.local` hostnames to `/etc/hosts` for host-based ingress routing.
- **Step 4 — Namespace**: Create the `mytravels-default` namespace that holds all resources.
- **Step 5 — PostgreSQL**: Deploy PostgreSQL 17.6 with secret, PVC, deployment, and ClusterIP service.
- **Step 6 — Database Migrations**: Run the once-off `db-migrations` Job (cleanup initContainer + the `efbundle` migrations executable).
- **Step 7 — RabbitMQ**: Deploy RabbitMQ with the management and Prometheus plugins enabled via ConfigMap.
- **Step 8 — MinIO**: Deploy MinIO object storage pinned to agent-2 with hostPath-backed PVs.
- **Step 9 — API**: Deploy the stateless ASP.NET Core REST API with its secret and service.
- **Step 10 — Messaging**: Deploy the stateless ASP.NET Core background worker that consumes RabbitMQ messages.
- **Step 11 — MCP**: Deploy the MCP server — the same service layer as the API, exposed as MCP tools over streamable HTTP for an MCP client/agent.
- **Step 12 — Web**: Deploy the React UI — a static Vite build served by nginx, with no secret and no PVC.
- **Step 13 — Observability**: Deploy the OTel Collector, Prometheus, Tempo, Grafana, and the postgres-exporter and cadvisor exporters.
- **Step 14 — Traefik Configuration**: Add the `postgres` TCP entrypoint to Traefik via `HelmChartConfig`.
- **Step 15 — Ingress**: Apply the ingress rules exposing RabbitMQ, MinIO, the API, Messaging, MCP, the Web UI, Grafana, Prometheus, the OTel Collector, and PostgreSQL through Traefik.
- **Step 16 — Full Stack Verification**: Confirm all pods, PVCs, services, ingresses, and URLs are healthy, that every Prometheus target is `up`, and that a trace reaches Tempo.
- **Step 17 — Diagnostics**: Pull logs and events per service when something misbehaves.
- **Step 18 — Seed Test Data (Optional)**: Bulk-upload a folder of geotagged photos as points of interest through the ingress, so the Web app and dashboards have real data. Skips itself if no photo folder is configured.
- **Step 19 — Teardown**: Delete all resources in reverse order and optionally the whole cluster.

## Services in this stage

| Tier | Services |
|---|---|
| Data | PostgreSQL, RabbitMQ, MinIO |
| Application | API, Messaging, MCP, Web |
| Observability | OTel Collector, Prometheus, Tempo, Grafana, postgres-exporter, cadvisor |

## The application architecture  

![architecture](images/architecture.png)

---

## Step 1 — Prerequisites

Rancher Desktop, k3d, kubectl, JupyterLab, and Freelens/OpenLens install notes: [macOS](<../1-install tools (macos).md>) · [Ubuntu](<../1-install tools (ubuntu).md>) · [Windows](<../1-install tools (windows).md>).

Once Rancher Desktop is installed, open it and ensure the container engine is running before continuing.

In [1]:
%%bash
echo "=== Docker ==="
docker --version
echo "=== k3d ==="
k3d --version
echo "=== kubectl ==="
kubectl version --client 2>/dev/null || kubectl version --client --short

=== Docker ===
Docker version 29.5.3-rd, build 5d9ffe3
=== k3d ===
k3d version v5.8.3
k3s version v1.33.6-k3s1 (default)
=== kubectl ===
Client Version: v1.34.1
Kustomize Version: v5.7.1


---

## Step 2 — Create the Cluster

![cluster](images/k8s%20components.drawio.png)

Creates a local k3d cluster with 2 control plane nodes and 4 worker nodes. Traefik is bundled automatically by k3s and serves as the ingress controller.

| Flag | Meaning |
|---|---|
| `-p "8080:80@loadbalancer"` | Maps `localhost:8080` → cluster port 80 (Traefik web entrypoint) |
| `-p "5432:5432@loadbalancer"` | Maps `localhost:5432` → cluster port 5432 (Traefik postgres TCP entrypoint) |
| `--image ghcr.io/k3s-io/k3s:v1.35.3-k3s1` | Pins the k3s version for reproducible cluster creation |
| `--servers 2` | 2 control plane nodes |
| `--agents 4` | 4 worker nodes |

Node count matters twice later: MinIO (Step 8) is pinned to `k3d-mytravels-agent-2` by `nodeSelector`, so at least 3 agents must exist; and cadvisor (Step 13) is a DaemonSet, so you get one cadvisor pod — and one Prometheus target — per node, all 6 of them.

> Every port the browser uses goes through `8080`, because that is the only HTTP port mapped into the cluster. There is no second mapping per service — Traefik routes by hostname on that one port.

> Skip this cell if the cluster already exists (`k3d cluster list`).

> **Ensure Rancher Desktop is running before this cell.** k3d creates the cluster's nodes as Docker containers, so it needs a live Docker daemon — on Linux there's no system Docker install, Rancher Desktop *is* the daemon. If it isn't running (or hasn't finished starting its VM yet), `k3d cluster create` fails immediately with `Cannot connect to the Docker daemon at unix:///home/<user>/.rd/docker.sock`, because that socket file doesn't exist until Rancher Desktop creates it. Open Rancher Desktop and wait for it to fully start, then confirm with `docker info` before retrying.

In [5]:
%%bash
k3d cluster list
echo ""
if k3d cluster list -o json | grep -q '"name":"mytravels"'; then
  echo "A cluster named 'mytravels' already exists."
  echo "Either reuse it (skip the create cell) or delete it first:"
  echo "    k3d cluster delete mytravels"
else
  echo "No 'mytravels' cluster — safe to create."
fi

NAME        SERVERS   AGENTS   LOADBALANCER
mytravels   2/2       4/4      true

A cluster named 'mytravels' already exists.
Either reuse it (skip the create cell) or delete it first:
    k3d cluster delete mytravels


In [3]:
%%bash
k3d cluster create mytravels \
  -p "8080:80@loadbalancer" \
  -p "5432:5432@loadbalancer" \
  --image ghcr.io/k3s-io/k3s:v1.35.3-k3s1 \
  --servers 2 \
  --agents 4

INFO[0000] portmapping '5432:5432' targets the loadbalancer: defaulting to [servers:*:proxy agents:*:proxy] 
INFO[0000] portmapping '8080:80' targets the loadbalancer: defaulting to [servers:*:proxy agents:*:proxy] 
INFO[0000] Prep: Network                                
INFO[0000] Created network 'k3d-mytravels'              
INFO[0000] Created image volume k3d-mytravels-images    
INFO[0000] Starting new tools node...                   
INFO[0000] Creating initializing server node            
INFO[0000] Creating node 'k3d-mytravels-server-0'       
INFO[0000] Starting node 'k3d-mytravels-tools'          
INFO[0001] Creating node 'k3d-mytravels-server-1'       
INFO[0001] Creating node 'k3d-mytravels-agent-0'        
INFO[0001] Creating node 'k3d-mytravels-agent-1'        
INFO[0001] Creating node 'k3d-mytravels-agent-2'        
INFO[0001] Creating node 'k3d-mytravels-agent-3'        


WARN[0001] You're creating 2 server nodes: Please consider creating at least 3 to achieve etcd quorum & fault tolerance 


INFO[0001] Creating LoadBalancer 'k3d-mytravels-serverlb' 
INFO[0001] Using the k3d-tools node to gather environment information 
INFO[0001] HostIP: using network gateway 172.26.0.1 address 
INFO[0001] Starting cluster 'mytravels'                 
INFO[0001] Starting the initializing server...          
INFO[0001] Starting node 'k3d-mytravels-server-0'       
INFO[0003] Starting servers...                          
INFO[0003] Starting node 'k3d-mytravels-server-1'       
INFO[0020] Starting agents...                           
INFO[0020] Starting node 'k3d-mytravels-agent-1'        
INFO[0020] Starting node 'k3d-mytravels-agent-3'        
INFO[0020] Starting node 'k3d-mytravels-agent-0'        
INFO[0020] Starting node 'k3d-mytravels-agent-2'        
INFO[0023] Starting helpers...                          
INFO[0023] Starting node 'k3d-mytravels-serverlb'       
INFO[0030] Injecting records for hostAliases (incl. host.k3d.internal) and for 7 network members into CoreDNS configmap... 
I

In [4]:
%%bash
echo "=== Nodes ==="
kubectl get nodes
echo ""
echo "=== Traefik ==="
kubectl get pods -n kube-system -l app.kubernetes.io/name=traefik

=== Nodes ===
NAME                     STATUS   ROLES                AGE   VERSION
k3d-mytravels-agent-0    Ready    <none>               73s   v1.35.3+k3s1
k3d-mytravels-agent-1    Ready    <none>               73s   v1.35.3+k3s1
k3d-mytravels-agent-2    Ready    <none>               73s   v1.35.3+k3s1
k3d-mytravels-agent-3    Ready    <none>               73s   v1.35.3+k3s1
k3d-mytravels-server-0   Ready    control-plane,etcd   82s   v1.35.3+k3s1
k3d-mytravels-server-1   Ready    control-plane,etcd   76s   v1.35.3+k3s1

=== Traefik ===
NAME                       READY   STATUS    RESTARTS   AGE
traefik-59449f8f96-t845b   1/1     Running   0          50s


---

## Step 3 — /etc/hosts

The ingress rules in `9-ingress.yaml` use host-based routing. Add the entries below to your hosts file so your browser resolves the hostnames to localhost. Every host in `9-ingress.yaml` needs an entry — miss one and that service is unreachable even though the pod is healthy.

```
127.0.0.1  rabbitmq.mytravels.local
127.0.0.1  minio.mytravels.local
127.0.0.1  api.mytravels.local
127.0.0.1  messaging.mytravels.local
127.0.0.1  mcp.mytravels.local
127.0.0.1  web.mytravels.local
127.0.0.1  grafana.mytravels.local
127.0.0.1  prometheus.mytravels.local
127.0.0.1  otel.mytravels.local
```

Run **one** of the next two cells depending on your OS:

- **macOS/Linux** — appends to `/etc/hosts` via `sudo`, prompting for your password.
- **Windows** — appends to `C:\Windows\System32\drivers\etc\hosts`. There's no `sudo` on Windows, so instead the cell checks whether it has Administrator privileges and writes directly if so. If not, close Jupyter/VS Code and relaunch it "as Administrator", then re-run the cell.

### macOS/Linux

In [6]:
import subprocess
import getpass

password = getpass.getpass("sudo password: ")

hosts = [
    "rabbitmq.mytravels.local",
    "minio.mytravels.local",
    "api.mytravels.local",
    "messaging.mytravels.local",
    "mcp.mytravels.local",
    "web.mytravels.local",
    "grafana.mytravels.local",
    "prometheus.mytravels.local",
    "otel.mytravels.local",
]

with open("/etc/hosts", "r") as f:
    current = f.read()

for host in hosts:
    if host in current:
        print(f"Already present: {host}")
    else:
        entry = f"127.0.0.1  {host}\n"
        result = subprocess.run(
            ["sudo", "-S", "tee", "-a", "/etc/hosts"],
            input=f"{password}\n{entry}",
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            print(f"Added: {host}")
        else:
            print(f"Failed: {host} — {result.stderr.strip()}")

Already present: rabbitmq.mytravels.local
Already present: minio.mytravels.local
Already present: api.mytravels.local
Already present: messaging.mytravels.local
Already present: mcp.mytravels.local
Already present: web.mytravels.local
Already present: grafana.mytravels.local
Already present: prometheus.mytravels.local
Already present: otel.mytravels.local


### Windows

In [ ]:
import ctypes

hosts_path = r"C:\Windows\System32\drivers\etc\hosts"
hosts = [
    "rabbitmq.mytravels.local",
    "minio.mytravels.local",
    "api.mytravels.local",
    "messaging.mytravels.local",
    "mcp.mytravels.local",
    "web.mytravels.local",
    "grafana.mytravels.local",
    "prometheus.mytravels.local",
    "otel.mytravels.local",
]

def is_admin():
    try:
        return bool(ctypes.windll.shell32.IsUserAnAdmin())
    except Exception:
        return False

if not is_admin():
    print("Not running as Administrator — the hosts file is not writable.")
    print("Close Jupyter/VS Code and relaunch it via 'Run as Administrator', then re-run this cell.")
else:
    with open(hosts_path, "r") as f:
        current = f.read()

    with open(hosts_path, "a") as f:
        for host in hosts:
            if host in current:
                print(f"Already present: {host}")
            else:
                f.write(f"127.0.0.1  {host}\n")
                print(f"Added: {host}")

---

## Step 4 — Namespace

All MyTravels resources live in the `mytravels-default` namespace. This must exist before any service manifests are applied.

In [7]:
%%bash
kubectl apply -f manifests/1-namespace.yaml

namespace/mytravels-default created


In [8]:
%%bash
kubectl get namespace mytravels-default

NAME                STATUS   AGE
mytravels-default   Active   12s


---

## Step 5 — PostgreSQL

Deploys PostgreSQL 17.6. The secret keys (`POSTGRES_USER`, `POSTGRES_PASSWORD`, `POSTGRES_DB`) match the docker-compose environment variable names exactly.

| File | Creates |
|---|---|
| `1-secret.yaml` | `postgres-secret` — DB credentials |
| `2-pvc.yaml` | `postgres-data-pvc` — 200Mi data volume |
| `3-deployment.yaml` | `postgres` deployment with liveness probe |
| `4-service.yaml` | `postgres` ClusterIP service on 5432 |

External access is via the Traefik `IngressRouteTCP` in `9-ingress.yaml`, or `kubectl port-forward svc/postgres 5432:5432 -n mytravels-default` for direct local access.

> **Before applying:** update `1-secret.yaml` with real base64-encoded values if needed:
> ```bash
> echo -n 'your-value' | base64
> ```

In [9]:
%%bash
kubectl apply -f manifests/postgres

secret/postgres-secret created
persistentvolumeclaim/postgres-data-pvc created
deployment.apps/postgres created
service/postgres created


In [10]:
%%bash
kubectl rollout status deployment/postgres -n mytravels-default
echo ""
kubectl get pods,pvc,svc -n mytravels-default -l app=postgres

Waiting for deployment "postgres" rollout to finish: 0 of 1 updated replicas are available...
deployment "postgres" successfully rolled out

NAME                            READY   STATUS    RESTARTS   AGE
pod/postgres-65ddd68fc5-x224p   1/1     Running   0          26s

NAME               TYPE        CLUSTER-IP     EXTERNAL-IP   PORT(S)    AGE
service/postgres   ClusterIP   10.43.144.19   <none>        5432/TCP   26s


---

## Step 6 — Database Migrations

Runs two once-off tasks in sequence. The `cleanup-migrations` initContainer deletes specific rows from `EFMigrationsHistory`, then the `migrate-core-db` main container runs the `efbundle` migrations executable against `CoreDbContext`. The Job completes once — it is never restarted (`restartPolicy: Never`).

| docker-compose service | Kubernetes equivalent |
|---|---|
| `cleanup-migrations` | `initContainer: cleanup-migrations` in the `db-migrations` Job |
| `migrate-core-db` | main container in the `db-migrations` Job |
| `depends_on: postgres: service_healthy` | `pg_isready` loop inside `cleanup-migrations` |
| `restart: "no"` | `restartPolicy: Never` on the Job pod |

| File | Creates |
|---|---|
| `1-secret.yaml` | `migrations-secret` — `ConnectionStrings__CoreDbContext` |
| `2-job.yaml` | `db-migrations` Job |

Postgres credentials (`POSTGRES_USER`, `POSTGRES_PASSWORD`, `POSTGRES_DB`) are pulled from the existing `postgres-secret`.

> **Before applying:** populate `migrations/1-secret.yaml` with a real base64-encoded connection string:
> ```bash
> echo -n "Host=postgres;Port=5432;Database=CoreDb;Username=...;Password=..." | base64
> ```
> Replace `<base64-encoded-connection-string>` in `1-secret.yaml` with the output.

In [11]:
%%bash
# kubectl delete job db-migrations -n mytravels-default --wait=true
kubectl apply -f manifests/migrations/

secret/migrations-secret created
job.batch/db-migrations created


In [12]:
%%bash
# Poll until the pod is scheduled (handles the case where the cell runs before the pod exists)
until kubectl get pod -l job-name=db-migrations -n mytravels-default 2>/dev/null | grep -q db-migrations; do
  echo "Waiting for pod to be scheduled..."; sleep 2
done

# Wait until the init container finishes (pod moves past PodInitializing)
kubectl wait pod -l job-name=db-migrations -n mytravels-default \
  --for=condition=Initialized --timeout=120s

# Wait for the job to complete — guarantees migrate-core-db has started and exited
kubectl wait job/db-migrations -n mytravels-default \
  --for=condition=Complete --timeout=300s

echo "--LIST JOBS--"
kubectl get job db-migrations -n mytravels-default
echo "--CLEANUP MIGRATION LOGS--"
kubectl logs -n mytravels-default -l job-name=db-migrations -c cleanup-migrations
echo "--MIGRATION LOGS--"
kubectl logs -n mytravels-default -l job-name=db-migrations -c migrate-core-db

pod/db-migrations-7sf6j condition met
job.batch/db-migrations condition met
--LIST JOBS--
NAME            STATUS     COMPLETIONS   DURATION   AGE
db-migrations   Complete   1/1           32s        32s
--CLEANUP MIGRATION LOGS--
postgres:5432 - accepting connections
DO
--MIGRATION LOGS--
Failed executing DbCommand (8ms) [Parameters=[], CommandType='Text', CommandTimeout='30']
SELECT "MigrationId", "ProductVersion"
FROM config."EFMigrationsHistory"
ORDER BY "MigrationId";
Acquiring an exclusive lock for migration application. See https://aka.ms/efcore-docs-migrations-lock for more information if this takes too long.
Applying migration '20260806103818_Init'.
Applying migration '20590930075747_UpdateStoredProcedure'.
Applying migration '20600930075821_SeedData'.
Done.


---

## Step 7 — RabbitMQ

Deploys RabbitMQ 3 with the management and Prometheus plugins. `2-configmap.yaml` supplies the `enabled_plugins` file (`[rabbitmq_management,rabbitmq_prometheus].`) that activates them — without it RabbitMQ starts with 0 plugins, the management UI never comes up, and Prometheus has nothing to scrape on 15692.

`kubectl apply -f <dir>` walks a directory in filename order, which is what the numeric prefixes are for: the ConfigMap has to exist before the Deployment that mounts it, and the Secret and PVC before the pod that consumes them.

| File | Creates |
|---|---|
| `1-secret.yaml` | `rabbitmq-secret` — broker credentials |
| `2-configmap.yaml` | `rabbitmq-config` — `enabled_plugins` file |
| `3-pvc.yaml` | `rabbitmq-data-pvc` (200Mi) |
| `4-deployment.yaml` | `rabbitmq` deployment |
| `5-service.yaml` | `rabbitmq` (ClusterIP — AMQP 5672 + metrics 15692) + `rabbitmq-management` (ClusterIP 15672) |

| Port | Purpose |
|---|---|
| 5672 | AMQP — api and messaging connect here |
| 15672 | Management UI — exposed via Ingress (Step 15) |
| 15692 | Prometheus metrics — scraped in-cluster by the `rabbitmq` job (Step 13) |

The management UI is accessible at [http://rabbitmq.mytravels.local:8080](http://rabbitmq.mytravels.local:8080) via the Traefik Ingress (Step 15).

In [13]:
%%bash
kubectl apply -f manifests/rabbitmq/

secret/rabbitmq-secret created
configmap/rabbitmq-config created
persistentvolumeclaim/rabbitmq-data-pvc created
persistentvolumeclaim/rabbitmq-config-pvc created
deployment.apps/rabbitmq created
service/rabbitmq created
service/rabbitmq-management created


In [14]:
%%bash
kubectl rollout status deployment/rabbitmq -n mytravels-default
echo ""
kubectl get pods,pvc,svc -n mytravels-default -l app=rabbitmq

Waiting for deployment "rabbitmq" rollout to finish: 0 of 1 updated replicas are available...
deployment "rabbitmq" successfully rolled out

NAME                            READY   STATUS    RESTARTS   AGE
pod/rabbitmq-5dd6688466-qglhq   1/1     Running   0          26s

NAME                          TYPE        CLUSTER-IP      EXTERNAL-IP   PORT(S)              AGE
service/rabbitmq              ClusterIP   10.43.94.254    <none>        5672/TCP,15692/TCP   26s
service/rabbitmq-management   ClusterIP   10.43.211.176   <none>        15672/TCP            26s


In [15]:
%%bash
# Confirm both plugins loaded. 'completed with 0 plugins' means the ConfigMap did not mount.
POD=$(kubectl get pod -n mytravels-default -l app=rabbitmq -o jsonpath='{.items[0].metadata.name}')
kubectl logs -n mytravels-default "$POD" --tail=40 | grep -E 'plugin|completed'
echo ""
echo "=== Enabled plugins ==="
kubectl exec -n mytravels-default "$POD" -- rabbitmq-plugins list -e

2026-08-23 17:27:37.254695+00:00 [info] <0.750.0> Management plugin: HTTP (non-TLS) listener started on port 15672
 completed with 5 plugins.
2026-08-23 17:27:37.369356+00:00 [info] <0.684.0> Server startup complete; 5 plugins started.

=== Enabled plugins ===
Listing plugins with pattern ".*" ...
 Configured: E = explicitly enabled; e = implicitly enabled
 | Status: * = running on rabbit@rabbitmq-5dd6688466-qglhq
 |/
[e*] rabbitmq_federation       3.13.7
[E*] rabbitmq_management       3.13.7
[e*] rabbitmq_management_agent 3.13.7
[E*] rabbitmq_prometheus       3.13.7
[e*] rabbitmq_web_dispatch     3.13.7


---

## Step 8 — MinIO

Deploys MinIO (S3-compatible object storage). The deployment uses a `nodeSelector` pinned to `k3d-mytravels-agent-2` and manual PVs with `hostPath` mounts on that node.

| File | Creates |
|---|---|
| `1-secret.yaml` | `minio-secret` — root credentials |
| `2-pv-pvc.yaml` | PVs + PVCs for data (200Mi) and config (50Mi) |
| `3-deployment.yaml` | `minio` deployment pinned to agent-2 |
| `4-service.yaml` | `minio` (ClusterIP 9000) + `minio-console` (ClusterIP 9090) |

The deployment also sets `MINIO_PROMETHEUS_AUTH_TYPE: public`, which lets Prometheus scrape `minio:9000/minio/v2/metrics/cluster` without a bearer token (Step 13). Fine for a local teaching cluster; in production you would issue MinIO a scrape token instead.

In [16]:
%%bash
kubectl apply -f manifests/minio/

secret/minio-secret created
persistentvolume/minio-data-pv created
persistentvolume/minio-config-pv created
persistentvolumeclaim/minio-data-pvc created
persistentvolumeclaim/minio-config-pvc created
deployment.apps/minio created
service/minio created
service/minio-console created


In [17]:
%%bash
kubectl rollout status deployment/minio -n mytravels-default
echo ""
kubectl get pods,pvc,svc -n mytravels-default -l app=minio

Waiting for deployment "minio" rollout to finish: 0 of 1 updated replicas are available...
deployment "minio" successfully rolled out

NAME                         READY   STATUS    RESTARTS   AGE
pod/minio-646bcc5949-cjcg6   1/1     Running   0          17s

NAME                    TYPE        CLUSTER-IP      EXTERNAL-IP   PORT(S)    AGE
service/minio           ClusterIP   10.43.103.211   <none>        9000/TCP   17s
service/minio-console   ClusterIP   10.43.224.80    <none>        9090/TCP   17s


---

## Step 9 — API

Deploys the MyTravels ASP.NET Core REST API. The service is stateless — no PVC is required.

Secret keys match the docker-compose `environment` keys exactly. Non-sensitive config (`ASPNETCORE_ENVIRONMENT`, `ASPNETCORE_URLS`, public URLs, `MinIO__Endpoint`, the OTel settings) is set directly in the Deployment rather than in the Secret.

| docker-compose env var | Kubernetes |
|---|---|
| `ConnectionStrings__CoreDbContext` | Secret `api-secret` → `secretKeyRef` |
| `GoogleApiKey` | Secret `api-secret` → `secretKeyRef` |
| `RabbitMQ__Uri` | Secret `api-secret` → `secretKeyRef` |
| `MinIO__AccessKey` | Secret `api-secret` → `secretKeyRef` |
| `MinIO__SecretKey` | Secret `api-secret` → `secretKeyRef` |
| `ASPNETCORE_ENVIRONMENT` | Direct env var `Production` |
| `ASPNETCORE_URLS` | Direct env var `http://+:5101` |
| `GoogleMapsUrl` | Direct env var `https://maps.googleapis.com` |
| `GooglePlacesUrl` | Direct env var `https://places.googleapis.com` |
| `MinIO__Endpoint` | Direct env var `minio:9000` (in-cluster DNS) |
| `OTEL_EXPORTER_OTLP_ENDPOINT` | Direct env var `http://otel-collector:4317` (Step 13) |
| `OTEL_SERVICE_NAME` | Direct env var `mytravels-api` (Step 13) |

| File | Creates |
|---|---|
| `1-secret.yaml` | `api-secret` — credentials and tokens |
| `2-deployment.yaml` | `api` deployment with liveness probe on `/health` |
| `3-service.yaml` | `api` ClusterIP service on 5101 |

The API is accessible at [http://api.mytravels.local:8080](http://api.mytravels.local:8080) via the Traefik Ingress (Step 15).

> **The two `OTEL_*` vars are what make this service show up in Grafana** — they point the .NET OpenTelemetry SDK at the collector deployed in Step 13, and `OTEL_SERVICE_NAME` is the `service.name` you filter on when searching traces in Tempo. The API pod starts fine without the collector running; traces are simply dropped until it exists.
>
> **Before applying:** populate `api/1-secret.yaml` with real base64-encoded values:
> ```bash
> echo -n "your-value" | base64
> ```
>
> **Dependency:** RabbitMQ (Step 7) must be healthy and the migrations Job (Step 6) must have completed successfully before applying the API Deployment. The `depends_on` from docker-compose has no automatic equivalent in Kubernetes.

In [18]:
%%bash
kubectl apply -f manifests/api/

secret/api-secret created
deployment.apps/api created
service/api created


In [19]:
%%bash
kubectl rollout status deployment/api -n mytravels-default
echo ""
kubectl get pods,svc -n mytravels-default -l app=api

Waiting for deployment "api" rollout to finish: 0 of 1 updated replicas are available...
deployment "api" successfully rolled out

NAME                       READY   STATUS    RESTARTS   AGE
pod/api-5c66554fdd-6m596   1/1     Running   0          12s

NAME          TYPE        CLUSTER-IP     EXTERNAL-IP   PORT(S)    AGE
service/api   ClusterIP   10.43.68.170   <none>        5101/TCP   12s


---

## Step 10 — Messaging

Deploys the MyTravels ASP.NET Core background worker that consumes RabbitMQ messages. The service is stateless — no PVC is required.

Secret keys match the docker-compose `environment` keys exactly. Non-sensitive config (`ASPNETCORE_ENVIRONMENT`, `ASPNETCORE_URLS`, public URLs, `MinIO__Endpoint`, the OTel settings) is set directly in the Deployment. Shared secrets (`ConnectionStrings__CoreDbContext`, `RabbitMQ__Uri`, `MinIO__AccessKey`, `MinIO__SecretKey`, `GoogleApiKey`) use the same base64 values as `api-secret` but are stored in the dedicated `messaging-secret`. `ContentSafetyEndpoint` and `ContentSafetyKey` are new keys not present in the API.

| docker-compose env var | Kubernetes |
|---|---|
| `ConnectionStrings__CoreDbContext` | Secret `messaging-secret` → `secretKeyRef` |
| `RabbitMQ__Uri` | Secret `messaging-secret` → `secretKeyRef` |
| `MinIO__AccessKey` | Secret `messaging-secret` → `secretKeyRef` |
| `MinIO__SecretKey` | Secret `messaging-secret` → `secretKeyRef` |
| `GoogleApiKey` | Secret `messaging-secret` → `secretKeyRef` |
| `ContentSafetyEndpoint` | Secret `messaging-secret` → `secretKeyRef` |
| `ContentSafetyKey` | Secret `messaging-secret` → `secretKeyRef` |
| `ASPNETCORE_ENVIRONMENT` | Direct env var `Production` |
| `ASPNETCORE_URLS` | Direct env var `http://+:5102` |
| `GoogleMapsUrl` | Direct env var `https://maps.googleapis.com` |
| `GooglePlacesUrl` | Direct env var `https://places.googleapis.com` |
| `MinIO__Endpoint` | Direct env var `minio:9000` (in-cluster DNS) |
| `OTEL_EXPORTER_OTLP_ENDPOINT` | Direct env var `http://otel-collector:4317` (Step 13) |
| `OTEL_SERVICE_NAME` | Direct env var `mytravels-messaging` (Step 13) |

| File | Creates |
|---|---|
| `1-secret.yaml` | `messaging-secret` — credentials and tokens |
| `2-deployment.yaml` | `messaging` deployment with liveness probe on `/health` |
| `3-service.yaml` | `messaging` ClusterIP service on 5102 |

The messaging worker has no user-facing UI, but it is exposed via Ingress at [http://messaging.mytravels.local:8080/health](http://messaging.mytravels.local:8080/health) for health checks (Step 15). Its work is observed through Grafana instead — traces under `service.name=mytravels-messaging` in Tempo, and queue depth via the RabbitMQ metrics Prometheus scrapes on 15692.

> **Before applying:** populate `messaging/1-secret.yaml` with real base64-encoded values for `ContentSafetyEndpoint` and `ContentSafetyKey`:
> ```bash
> echo -n "https://your-endpoint.cognitiveservices.azure.com/" | base64   # ContentSafetyEndpoint
> echo -n "your-key" | base64                                              # ContentSafetyKey
> ```
>
> **Dependency:** RabbitMQ (Step 7) must be healthy and MinIO (Step 8) must be running before the messaging worker can process messages.

In [20]:
%%bash
kubectl apply -f manifests/messaging/

secret/messaging-secret created
deployment.apps/messaging created
service/messaging created


In [21]:
%%bash
kubectl rollout status deployment/messaging -n mytravels-default
echo ""
kubectl get pods,svc -n mytravels-default -l app=messaging

Waiting for deployment "messaging" rollout to finish: 0 of 1 updated replicas are available...
deployment "messaging" successfully rolled out

NAME                             READY   STATUS    RESTARTS   AGE
pod/messaging-76969859b6-j7gw9   1/1     Running   0          19s

NAME                TYPE        CLUSTER-IP      EXTERNAL-IP   PORT(S)    AGE
service/messaging   ClusterIP   10.43.198.182   <none>        5102/TCP   19s


---

## Step 11 — MCP

Deploys the MyTravels MCP server — exposes `upload_photo`, `upload_photo_with_coordinates`, and `search_place` as MCP tools over streamable HTTP, reusing the same service layer as the API rather than proxying it. The service is stateless — no PVC is required.

Secret keys match the docker-compose `environment` keys exactly, and reuse the same base64 values as `api-secret`. Non-sensitive config is set directly in the Deployment, same pattern as API/Messaging. Unlike API and Messaging, there is no `GoogleApiKey` entry at all — MCP's `search_place` tool still calls `IMapsService`, but geocoding falls back to OpenStreetMap without it, matching the API's existing behavior in this stage (SPEC F-14).

| docker-compose env var | Kubernetes |
|---|---|
| `ConnectionStrings__CoreDbContext` | Secret `mcp-secret` → `secretKeyRef` |
| `RabbitMQ__Uri` | Secret `mcp-secret` → `secretKeyRef` |
| `MinIO__AccessKey` | Secret `mcp-secret` → `secretKeyRef` |
| `MinIO__SecretKey` | Secret `mcp-secret` → `secretKeyRef` |
| `ASPNETCORE_ENVIRONMENT` | Direct env var `Production` |
| `ASPNETCORE_URLS` | Direct env var `http://+:5103` |
| `GoogleMapsUrl` | Direct env var `https://maps.googleapis.com` |
| `GooglePlacesUrl` | Direct env var `https://places.googleapis.com` |
| `MinIO__Endpoint` | Direct env var `minio:9000` (in-cluster DNS) |
| `OTEL_EXPORTER_OTLP_ENDPOINT` | Direct env var `http://otel-collector:4317` (Step 13) |
| `OTEL_SERVICE_NAME` | Direct env var `mytravels-mcp` (Step 13) |

| File | Creates |
|---|---|
| `1-secret.yaml` | `mcp-secret` — credentials and tokens |
| `2-deployment.yaml` | `mcp` deployment with liveness probe on `/health` (no Swagger here, unlike API) |
| `3-service.yaml` | `mcp` ClusterIP service on 5103 |

**Endpoints.** `Program.cs` maps only two things: `GET /health` (used by the probes and by the verification cell in Step 16) and `MapMcp()` at the root path, which is the streamable-HTTP MCP transport — it speaks JSON-RPC over `POST /`, so opening it in a browser is not a useful check.

**Reaching it.** `9-ingress.yaml` (Step 15) routes `mcp.mytravels.local` → `mcp:5103`, so MCP is reachable both ways:

- in-cluster: `http://mcp:5103` — how another pod would call it
- from your machine: `http://mcp.mytravels.local:8080` — through Traefik, the same as every other host-routed service

> **`.mcp.json` in this directory points at `http://mcp.mytravels.local/` (port 80).** The k3d cluster maps host port **8080** → cluster port 80 (Step 2), so a client on your machine needs `http://mcp.mytravels.local:8080/`. Either fix the URL in `.mcp.json`, or recreate the cluster with `-p "80:80@loadbalancer"` if you want the bare hostname to work.

> **Before applying:** populate `mcp/1-secret.yaml` with real base64-encoded values — same as `api/1-secret.yaml`:
> ```bash
> echo -n "your-value" | base64
> ```
>
> **Dependency:** RabbitMQ (Step 7) must be healthy and the migrations Job (Step 6) must have completed successfully before applying the MCP Deployment — same dependency as the API.

In [22]:
%%bash
kubectl apply -f manifests/mcp/

secret/mcp-secret created
deployment.apps/mcp created
service/mcp created


In [23]:
%%bash
kubectl rollout status deployment/mcp -n mytravels-default
echo ""
kubectl get pods,svc -n mytravels-default -l app=mcp

Waiting for deployment "mcp" rollout to finish: 0 of 1 updated replicas are available...
deployment "mcp" successfully rolled out

NAME                       READY   STATUS    RESTARTS   AGE
pod/mcp-76d87689b8-2zwtz   1/1     Running   0          13s

NAME          TYPE        CLUSTER-IP     EXTERNAL-IP   PORT(S)    AGE
service/mcp   ClusterIP   10.43.20.138   <none>        5103/TCP   13s


---

## Step 12 — Web

Deploys the MyTravels React UI. The image is a static Vite build served by nginx (see [src/web/Dockerfile](<../src/web/Dockerfile>)) — no secret and no PVC, because the container serves files straight off its own image layer.

| docker-compose | Kubernetes |
|---|---|
| `image: mytravels-web:v1.0.4` | `tshepontlhokoa/mytravels-web:v1.0.4` |
| `ports: 5100:80` | `web` ClusterIP service on 80, reached through the Ingress (Step 15) |
| `build.args.VITE_API_BASE_URL` | **no equivalent** — see the note below |
| `depends_on: api` | no equivalent; the pod starts regardless and the browser's API calls fail until the API is up |

| File | Creates |
|---|---|
| `2-deployment.yaml` | `web` deployment with readiness and liveness probes on `/` |
| `3-service.yaml` | `web` ClusterIP service on 80 |

> **`VITE_API_BASE_URL` is a build-time argument, not a runtime one.** Vite inlines it into the JavaScript bundle when the image is built, so unlike every other service in this runbook there is no env var to set on the Deployment — the browser calls whatever host the image was built with. The published `v1.0.4` image was built with `http://localhost:5101`, so the UI only reaches the API while `kubectl port-forward svc/api 5101:5101 -n mytravels-default` is running. To go through the ingress instead, rebuild the image with `VITE_API_BASE_URL=http://api.mytravels.local:8080` and bump the tag in `web/2-deployment.yaml`.
>
> The usual fixes are to have the container's entrypoint write a `config.js` that the app reads at load time, or to serve the API under the same host on a `/api` path so a relative URL works. Both are out of scope here — this step is about the difference between build-time and runtime configuration, which is worth meeting once.


In [24]:
%%bash
kubectl apply -f manifests/web/

deployment.apps/web created
service/web created


In [25]:
%%bash
kubectl rollout status deployment/web -n mytravels-default
echo ""
kubectl get pods,svc -n mytravels-default -l app=web

Waiting for deployment "web" rollout to finish: 0 of 1 updated replicas are available...
deployment "web" successfully rolled out

NAME                       READY   STATUS    RESTARTS   AGE
pod/web-66bd67c4bc-7lw5p   1/1     Running   0          19s

NAME          TYPE        CLUSTER-IP     EXTERNAL-IP   PORT(S)   AGE
service/web   ClusterIP   10.43.217.87   <none>        80/TCP    19s


---

## Step 13 — Observability

Deploys metrics and distributed tracing for the whole stack. The API, messaging worker, and MCP server push OTLP metrics/traces to the OTel Collector, which exposes a Prometheus scrape endpoint and forwards traces to Tempo. Prometheus also scrapes RabbitMQ, MinIO, and postgres-exporter directly (they already speak the Prometheus exposition format), plus cadvisor for per-container/node metrics. Grafana ships with both datasources and one starter dashboard pre-provisioned.

```
api / messaging / mcp ──OTLP:4317──► otel-collector ──:8889 (scraped)──► prometheus ──► grafana
                                          └──────traces──────► tempo ──────────────────► grafana
rabbitmq :15692 ─┐
minio    :9000  ─┼──scraped──────────────────────────────────► prometheus
postgres-exporter :9187 ─┤
cadvisor :8080 (DaemonSet, one per node) ─┘
```

### Components

| Component | Image | Port(s) | Role |
|---|---|---|---|
| OTel Collector | `otel/opentelemetry-collector-contrib:0.116.1` | 4317 gRPC, 4318 HTTP, 8889 metrics | Receives OTLP from the .NET services; exposes `:8889` for Prometheus, forwards traces to Tempo |
| Prometheus | `prom/prometheus:v3.1.0` | 9090 | Scrapes every target below; 1Gi PVC for the TSDB |
| Tempo | `grafana/tempo:2.6.1` | 4317/4318 OTLP in, 3200 query API | Trace storage and query; 1Gi PVC |
| Grafana | `grafana/grafana:11.4.0` | 3000 | Dashboards over Prometheus + Tempo; 200Mi PVC; sign-up disabled |
| postgres-exporter | `prometheuscommunity/postgres-exporter:v0.15.0` | 9187 | PostgreSQL metrics — reuses `postgres-secret`, no new credential |
| cadvisor | `gcr.io/cadvisor/cadvisor:v0.49.1` | 8080 | Container/node resource metrics — **DaemonSet**, one pod per node |

### Prometheus scrape targets (`4-prometheus-configmap.yaml`)

| Job | Target | Source |
|---|---|---|
| `prometheus` | `localhost:9090` | itself |
| `otel-collector` | `otel-collector:8889` | api, messaging, mcp (via OTLP) |
| `postgres` | `postgres-exporter:9187` | PostgreSQL |
| `rabbitmq` | `rabbitmq:15692` | `rabbitmq_prometheus` plugin |
| `minio` | `minio:9000` | MinIO built-in `/minio/v2/metrics/cluster` |
| `cadvisor` | DNS `SRV` on the headless `cadvisor` Service | one entry per node, discovered dynamically |

cadvisor is the one job that isn't a fixed target: because it's a DaemonSet, the pod count changes with the node count, so Prometheus uses DNS `SRV` discovery against the headless (`clusterIP: None`) `cadvisor` Service instead of a hardcoded list. Add a node, and the new cadvisor pod is scraped without editing any config.

### Files

| File | Creates |
|---|---|
| `1-otel-collector-configmap.yaml` … `3-otel-collector-service.yaml` | ConfigMap, Deployment, Service |
| `4-prometheus-configmap.yaml` … `7-prometheus-service.yaml` | ConfigMap, PVC, Deployment, Service |
| `8-tempo-configmap.yaml` … `11-tempo-service.yaml` | ConfigMap, PVC, Deployment, Service |
| `12-postgres-exporter-deployment.yaml`, `13-postgres-exporter-service.yaml` | Deployment, Service |
| `14-cadvisor-daemonset.yaml`, `15-cadvisor-service.yaml` | DaemonSet, headless Service |
| `16-grafana-secret.yaml` … `21-grafana-service.yaml` | Secret, datasources ConfigMap, dashboards ConfigMap, PVC, Deployment, Service |

### Changes to earlier manifests

This step also changes manifests already applied in earlier steps — the cell below re-applies them:

- `rabbitmq/2-configmap.yaml` — enables the `rabbitmq_prometheus` plugin
- `rabbitmq/4-deployment.yaml` / `5-service.yaml` — expose port `15692`
- `minio/3-deployment.yaml` — adds `MINIO_PROMETHEUS_AUTH_TYPE: public` (otherwise the metrics endpoint requires a bearer token)
- `api/2-deployment.yaml` / `messaging/2-deployment.yaml` — add `OTEL_EXPORTER_OTLP_ENDPOINT` and `OTEL_SERVICE_NAME`

MCP's `2-deployment.yaml` (Step 11) already ships with `OTEL_EXPORTER_OTLP_ENDPOINT`/`OTEL_SERVICE_NAME` baked in from creation, so it isn't part of this re-apply list — there's nothing to retrofit.

> **cadvisor needs privileged access to the node** (`hostPath` mounts of `/`, `/sys`, `/var/run`, `/var/lib/docker`) to read container metrics — this is normal for a node-level metrics agent, not a misconfiguration.

In [26]:
%%bash
kubectl apply -f manifests/observability/
echo ""
echo "=== Re-applying manifests changed for observability ==="
kubectl apply -f manifests/rabbitmq/
kubectl apply -f manifests/minio/
kubectl apply -f manifests/api/
kubectl apply -f manifests/messaging/

configmap/otel-collector-config created
deployment.apps/tempo created
service/tempo created
deployment.apps/postgres-exporter created
service/postgres-exporter created
daemonset.apps/cadvisor created
service/cadvisor created
secret/grafana-secret created
configmap/grafana-datasources created
configmap/grafana-dashboards created
persistentvolumeclaim/grafana-data-pvc created
deployment.apps/otel-collector created
deployment.apps/grafana created
service/grafana created
service/otel-collector created
configmap/prometheus-config created
persistentvolumeclaim/prometheus-data-pvc created
deployment.apps/prometheus created
service/prometheus created
configmap/tempo-config created
persistentvolumeclaim/tempo-data-pvc created

=== Re-applying manifests changed for observability ===
secret/rabbitmq-secret unchanged
configmap/rabbitmq-config unchanged
persistentvolumeclaim/rabbitmq-data-pvc unchanged
persistentvolumeclaim/rabbitmq-config-pvc unchanged
deployment.apps/rabbitmq unchanged
service/ra

In [27]:
%%bash
kubectl rollout status deployment/otel-collector -n mytravels-default
kubectl rollout status deployment/prometheus -n mytravels-default
kubectl rollout status deployment/tempo -n mytravels-default
kubectl rollout status deployment/postgres-exporter -n mytravels-default
kubectl rollout status daemonset/cadvisor -n mytravels-default
kubectl rollout status deployment/grafana -n mytravels-default
echo ""
kubectl get pods,pvc,svc -n mytravels-default -l 'app in (otel-collector,prometheus,tempo,postgres-exporter,cadvisor,grafana)'

Waiting for deployment "otel-collector" rollout to finish: 0 of 1 updated replicas are available...
deployment "otel-collector" successfully rolled out
Waiting for deployment "prometheus" rollout to finish: 0 of 1 updated replicas are available...
deployment "prometheus" successfully rolled out
Waiting for deployment "tempo" rollout to finish: 0 of 1 updated replicas are available...
deployment "tempo" successfully rolled out
deployment "postgres-exporter" successfully rolled out
daemon set "cadvisor" successfully rolled out
deployment "grafana" successfully rolled out

NAME                                     READY   STATUS    RESTARTS   AGE
pod/cadvisor-66klt                       1/1     Running   0          85s
pod/cadvisor-7qx5j                       1/1     Running   0          85s
pod/cadvisor-9x2rp                       1/1     Running   0          85s
pod/cadvisor-bt5gz                       1/1     Running   0          85s
pod/cadvisor-dkhcs                       1/1     Runn

---

## Step 14 — Traefik Configuration

Adds the `postgres` TCP entrypoint to Traefik so that the `IngressRouteTCP` in `9-ingress.yaml` can route raw TCP connections on port 5432 to the postgres pod. Without this, connections to `127.0.0.1:5432` are refused even when postgres is healthy.

The `HelmChartConfig` patches the k3s-managed Traefik Helm release — k3s picks it up and restarts Traefik automatically within ~15 seconds.

| File | Creates |
|---|---|
| `8-traefik-config.yaml` | `HelmChartConfig/traefik` — adds `--entrypoints.postgres.address=:5432/tcp` |

> **Cluster port mapping:** the k3d cluster must have been created with `-p "5432:5432@loadbalancer"` (see Step 2) so that the Traefik entrypoint is reachable from the host.

In [28]:
%%bash
kubectl apply -f manifests/8-traefik-config.yaml
echo ""
echo "Waiting for Traefik to restart..."
sleep 20
kubectl rollout status deploy/traefik -n kube-system --timeout=60s
echo ""
for i in $(seq 1 12); do
  ARGS=$(kubectl get deploy traefik -n kube-system -o jsonpath='{.spec.template.spec.containers[0].args}' | tr ',' '\n')
  if echo "$ARGS" | grep -q postgres; then
    echo "postgres entrypoint confirmed:"
    echo "$ARGS" | grep postgres
    break
  fi
  echo "Waiting for postgres entrypoint... ($i/12)"
  sleep 5
done

helmchartconfig.helm.cattle.io/traefik created

Waiting for Traefik to restart...
Waiting for deployment "traefik" rollout to finish: 1 old replicas are pending termination...
Waiting for deployment "traefik" rollout to finish: 1 old replicas are pending termination...
deployment "traefik" successfully rolled out

postgres entrypoint confirmed:
"--entryPoints.postgres.address=:5432/tcp"


---

## Step 15 — Ingress

In [29]:
%%bash
kubectl apply -f manifests/9-ingress.yaml

ingress.networking.k8s.io/management-ingress created
ingress.networking.k8s.io/api-ingress created
ingress.networking.k8s.io/messaging-ingress created
ingress.networking.k8s.io/web-ingress created
ingress.networking.k8s.io/mcp-ingress created
ingressroutetcp.traefik.io/postgres-tcp-ingress created


In [30]:
%%bash
kubectl get ingress -n mytravels-default

NAME                 CLASS     HOSTS                                                                                ADDRESS                                                             PORTS   AGE
api-ingress          traefik   api.mytravels.local                                                                  172.26.0.2,172.26.0.3,172.26.0.4,172.26.0.5,172.26.0.6,172.26.0.7   80      6s
management-ingress   traefik   rabbitmq.mytravels.local,minio.mytravels.local,grafana.mytravels.local + 2 more...   172.26.0.2,172.26.0.3,172.26.0.4,172.26.0.5,172.26.0.6,172.26.0.7   80      6s
mcp-ingress          traefik   mcp.mytravels.local                                                                  172.26.0.2,172.26.0.3,172.26.0.4,172.26.0.5,172.26.0.6,172.26.0.7   80      6s
messaging-ingress    traefik   messaging.mytravels.local                                                            172.26.0.2,172.26.0.3,172.26.0.4,172.26.0.5,172.26.0.6,172.26.0.7   80      6s
web-ingress          tra


Applies the top-level ingress rules that expose services through Traefik.

| Host | Routes to | Port | Protocol |
|---|---|---|---|
| [http://rabbitmq.mytravels.local:8080](http://rabbitmq.mytravels.local:8080) | `rabbitmq-management` service | 15672 | HTTP |
| [http://minio.mytravels.local:8080](http://minio.mytravels.local:8080) | `minio-console` service | 9090 | HTTP |
| [http://api.mytravels.local:8080](http://api.mytravels.local:8080/swagger) | `api` service | 5101 | HTTP |
| [http://messaging.mytravels.local:8080/health](http://messaging.mytravels.local:8080/health) | `messaging` service | 5102 | HTTP |
| [http://mcp.mytravels.local:8080/health](http://mcp.mytravels.local:8080/health) | `mcp` service | 5103 | HTTP (MCP streamable HTTP at `/`) |
| [http://web.mytravels.local:8080](http://web.mytravels.local:8080) | `web` service | 80 | HTTP |
| [http://grafana.mytravels.local:8080](http://grafana.mytravels.local:8080) | `grafana` service | 3000 | HTTP |
| [http://prometheus.mytravels.local:8080](http://prometheus.mytravels.local:8080) | `prometheus` service | 9090 | HTTP |
| [http://otel.mytravels.local:8080](http://otel.mytravels.local:8080) | `otel-collector` service | 4318 | HTTP (OTLP, browser RUM) |
| `127.0.0.1:5432` (TCP) | `postgres` service | 5432 | TCP |

Three services deliberately have **no** ingress row and stay cluster-internal:

- **Tempo** (`tempo:3200`) — queried by Grafana through its datasource, not by you directly. Use Grafana's Explore view for traces.
- **postgres-exporter** (`postgres-exporter:9187`) and **cadvisor** (`cadvisor:8080`) — scrape endpoints Prometheus reads in-cluster; there's no UI to expose.

`9-ingress.yaml` defines five `Ingress` objects (`management-ingress` covering RabbitMQ/MinIO/Grafana/Prometheus/OTel, plus `api-ingress`, `messaging-ingress`, `web-ingress`, `mcp-ingress`) and one Traefik `IngressRouteTCP` for PostgreSQL.

Traffic flow (HTTP): `localhost:8080` → k3d load balancer → Traefik (port 80) → service backend.
PostgreSQL: `localhost:5432` → k3d load balancer → Traefik (postgres entrypoint) → postgres service.

---

## Step 16 — Full Stack Verification

Run these cells to confirm all resources are healthy before using the stack.

In [31]:
%%bash
echo "=== Nodes ==="
kubectl get nodes
echo ""
echo "=== Pods ==="
kubectl get pods -n mytravels-default -o wide
echo ""
echo "=== Deployments / DaemonSets / Jobs ==="
kubectl get deployments,daemonsets,jobs -n mytravels-default
echo ""
echo "=== PVCs ==="
kubectl get pvc -n mytravels-default
echo ""
echo "=== Services ==="
kubectl get svc -n mytravels-default
echo ""
echo "=== Ingress ==="
kubectl get ingress,ingressroutetcp -n mytravels-default
echo ""

# Any pod not Running/Completed gets its events dumped here, inline.
BAD=$(kubectl get pods -n mytravels-default --no-headers \
  | awk '$3 != "Running" && $3 != "Completed" {print $1}')
if [ -n "$BAD" ]; then
  echo "=== Unhealthy pods ==="
  for POD in $BAD; do
    echo "--- $POD ---"
    kubectl describe pod "$POD" -n mytravels-default | sed -n '/Events:/,$p'
  done
else
  echo "All pods Running or Completed."
fi

=== Nodes ===
NAME                     STATUS   ROLES                AGE   VERSION
k3d-mytravels-agent-0    Ready    <none>               49m   v1.35.3+k3s1
k3d-mytravels-agent-1    Ready    <none>               49m   v1.35.3+k3s1
k3d-mytravels-agent-2    Ready    <none>               49m   v1.35.3+k3s1
k3d-mytravels-agent-3    Ready    <none>               49m   v1.35.3+k3s1
k3d-mytravels-server-0   Ready    control-plane,etcd   49m   v1.35.3+k3s1
k3d-mytravels-server-1   Ready    control-plane,etcd   49m   v1.35.3+k3s1

=== Pods ===
NAME                                 READY   STATUS      RESTARTS   AGE     IP          NODE                     NOMINATED NODE   READINESS GATES
api-5c66554fdd-6m596                 1/1     Running     0          8m6s    10.42.7.3   k3d-mytravels-agent-0    <none>           <none>
cadvisor-66klt                       1/1     Running     0          4m41s   10.42.0.8   k3d-mytravels-server-0   <none>           <none>
cadvisor-7qx5j                       1/

In [37]:
%%bash
# Ingress-exposed services. A check fails on any non-2xx/3xx status (or no response
# at all), and the diagnostics for that service run immediately, inline.
check() {
  NAME=$1; URL=$2; LABEL=$3
  CODE=$(curl -s -o /dev/null -w "%{http_code}" --max-time 10 "$URL" || echo "000")
  case "$CODE" in
    2*|3*) printf "%-16s %s OK\n" "$NAME" "$CODE" ;;
    *)
      printf "%-16s %s FAILED (%s)\n" "$NAME" "$CODE" "$URL"
      echo "--- pods (app=$LABEL) ---"
      kubectl get pods -n mytravels-default -l app="$LABEL" -o wide
      echo "--- logs (app=$LABEL, last 20) ---"
      kubectl logs -n mytravels-default -l app="$LABEL" --tail=20 2>&1 | sed 's/^/    /'
      echo ""
      ;;
  esac
}

echo "=== Application tier ==="
check "Web"        http://web.mytravels.local:8080                 web
check "API"        http://api.mytravels.local:8080/swagger/index.html api
check "Messaging"  http://messaging.mytravels.local:8080/health    messaging
check "MCP"        http://mcp.mytravels.local:8080/health          mcp
echo ""
echo "=== Data tier ==="
check "RabbitMQ"   http://rabbitmq.mytravels.local:8080            rabbitmq
check "MinIO"      http://minio.mytravels.local:8080               minio
echo ""
echo "=== Observability tier ==="
check "Grafana"    http://grafana.mytravels.local:8080/api/health   grafana
check "Prometheus" http://prometheus.mytravels.local:8080/-/healthy prometheus
echo ""

# Cluster-internal services with no ingress route — one throwaway curl pod checks them all.
echo "=== Cluster-internal endpoints ==="
kubectl run cluster-healthcheck -n mytravels-default --rm -i --restart=Never \
  --image=curlimages/curl:8.11.0 --quiet -- sh -c '
for T in "tempo|http://tempo:3200/ready" \
         "otel-collector|http://otel-collector:8889/metrics" \
         "postgres-exporter|http://postgres-exporter:9187/metrics" \
         "cadvisor|http://cadvisor:8080/healthz" \
         "mcp (in-cluster)|http://mcp:5103/health"; do
  NAME=${T%%|*}; URL=${T#*|}
  printf "%-20s %s\n" "$NAME" "$(curl -s -o /dev/null -w "%{http_code}" --max-time 10 "$URL" || echo 000)"
done'

=== Application tier ===
Web              200 OK
API              200 OK
Messaging        200 OK
MCP              200 OK

=== Data tier ===
RabbitMQ         200 OK
MinIO            200 OK

=== Observability tier ===
Grafana          200 OK
Prometheus       200 OK

=== Cluster-internal endpoints ===
tempo                200
otel-collector       200
postgres-exporter    200
cadvisor             200
mcp (in-cluster)     200


In [33]:
%%bash
GRAFANA_AUTH="user123:password123"   # from observability/16-grafana-secret.yaml

echo "=== Prometheus scrape targets ==="
curl -s http://prometheus.mytravels.local:8080/api/v1/targets \
  | python3 -c "
import json, sys
data = json.load(sys.stdin)
targets = data['data']['activeTargets']
down = [t for t in targets if t['health'] != 'up']
for t in targets:
    print(f\"  {t['labels'].get('job'):<18} {t['health']:<8} {t['scrapeUrl']} {t.get('lastError','')}\")
print()
print(f'{len(targets) - len(down)}/{len(targets)} targets up')
jobs = {t['labels'].get('job') for t in targets}
missing = {'prometheus','otel-collector','postgres','rabbitmq','minio','cadvisor'} - jobs
if missing:
    print('MISSING JOBS:', ', '.join(sorted(missing)))
"
echo ""

echo "=== cadvisor: one target per node (DaemonSet) ==="
NODES=$(kubectl get nodes --no-headers | wc -l | tr -d ' ')
CADVISOR_PODS=$(kubectl get pods -n mytravels-default -l app=cadvisor --no-headers | grep -c Running)
CADVISOR_TARGETS=$(curl -s http://prometheus.mytravels.local:8080/api/v1/targets \
  | python3 -c "import json,sys; print(sum(1 for t in json.load(sys.stdin)['data']['activeTargets'] if t['labels'].get('job')=='cadvisor'))")
echo "nodes=$NODES  running cadvisor pods=$CADVISOR_PODS  prometheus cadvisor targets=$CADVISOR_TARGETS"
if [ "$NODES" != "$CADVISOR_PODS" ]; then
  echo "MISMATCH — DaemonSet is not on every node:"
  kubectl get pods -n mytravels-default -l app=cadvisor -o wide
  kubectl describe daemonset cadvisor -n mytravels-default | tail -20
fi
echo ""

echo "=== Generating traces (api, then mcp) ==="
curl -s -o /dev/null -w "  api HTTP %{http_code}\n" http://api.mytravels.local:8080/api/pointofinterest
curl -s -o /dev/null -w "  mcp HTTP %{http_code}\n" http://mcp.mytravels.local:8080/health
echo ""

echo "=== Waiting for traces to reach Tempo (via the Grafana datasource proxy) ==="
for SERVICE in mytravels-api mytravels-mcp; do
  FOUND=0
  for i in $(seq 1 12); do
    COUNT=$(curl -s -u "$GRAFANA_AUTH" \
      "http://grafana.mytravels.local:8080/api/datasources/proxy/uid/tempo/api/search?tags=service.name%3D${SERVICE}&limit=1" \
      | python3 -c "import json,sys; print(len(json.load(sys.stdin).get('traces', [])))" 2>/dev/null || echo 0)
    if [ "$COUNT" -gt 0 ]; then
      echo "  $SERVICE — trace found"
      FOUND=1
      break
    fi
    sleep 5
  done
  if [ "$FOUND" -eq 0 ]; then
    echo "  $SERVICE — NO TRACE after 60s"
    echo "  --- otel-collector logs ---"
    kubectl logs -n mytravels-default -l app=otel-collector --tail=20 | sed 's/^/      /'
    echo "  --- tempo logs ---"
    kubectl logs -n mytravels-default -l app=tempo --tail=20 | sed 's/^/      /'
  fi
done

=== Prometheus scrape targets ===
  cadvisor           up       http://10-42-7-4.cadvisor.mytravels-default.svc.cluster.local:8080/metrics 
  cadvisor           up       http://10-42-5-6.cadvisor.mytravels-default.svc.cluster.local:8080/metrics 
  cadvisor           up       http://10-42-3-6.cadvisor.mytravels-default.svc.cluster.local:8080/metrics 
  cadvisor           up       http://10-42-2-5.cadvisor.mytravels-default.svc.cluster.local:8080/metrics 
  cadvisor           up       http://10-42-6-6.cadvisor.mytravels-default.svc.cluster.local:8080/metrics 
  cadvisor           up       http://10-42-0-8.cadvisor.mytravels-default.svc.cluster.local:8080/metrics 
  minio              up       http://minio:9000/minio/v2/metrics/cluster 
  otel-collector     up       http://otel-collector:8889/metrics 
  postgres           up       http://postgres-exporter:9187/metrics 
  prometheus         up       http://localhost:9090/metrics 
  rabbitmq           up       http://rabbitmq:15692/metrics 

In [34]:
%%bash
kubectl top pods -A

NAMESPACE           NAME                                      CPU(cores)   MEMORY(bytes)   
kube-system         coredns-c4dbffb5f-wkcq4                   4m           20Mi            
kube-system         local-path-provisioner-5c4dc5d66d-l7kxh   1m           15Mi            
kube-system         metrics-server-786d997795-666kp           4m           27Mi            
kube-system         svclb-traefik-bc732a58-42kjb              0m           1Mi             
kube-system         svclb-traefik-bc732a58-9kpc7              0m           1Mi             
kube-system         svclb-traefik-bc732a58-bhr67              0m           1Mi             
kube-system         svclb-traefik-bc732a58-gpxsf              0m           1Mi             
kube-system         svclb-traefik-bc732a58-lwdxs              0m           1Mi             
kube-system         svclb-traefik-bc732a58-stm8j              0m           1Mi             
kube-system         traefik-7ff98c6b5b-4vcpb                  2m           24Mi 


**Services deployed:**

| Service | Workload | Purpose |
|---|---|---|
| PostgreSQL | Deployment | Primary database |
| RabbitMQ | Deployment | Message broker (management + Prometheus plugins) |
| MinIO | Deployment | S3-compatible object storage |
| db-migrations | Job | Once-off EF Core migrations (Step 6) |
| API | Deployment | ASP.NET Core REST API |
| Messaging | Deployment | ASP.NET Core background worker (RabbitMQ consumer) |
| MCP | Deployment | MCP server — same service layer as the API, exposed as MCP tools |
| Web | Deployment | React UI (static build served by nginx) |
| OTel Collector | Deployment | Receives OTLP metrics/traces, exports to Prometheus + Tempo |
| Prometheus | Deployment | Metrics storage and scraping |
| Tempo | Deployment | Distributed trace storage |
| Grafana | Deployment | Dashboards over Prometheus + Tempo |
| postgres-exporter | Deployment | PostgreSQL → Prometheus metrics bridge |
| cadvisor | **DaemonSet** | Per-node container resource metrics |

**Browser-reachable URLs (after full setup):**

| Service | URL | Credentials |
|---|---|---|
| Web UI | [http://web.mytravels.local:8080](http://web.mytravels.local:8080) | — |
| API (Swagger) | [http://api.mytravels.local:8080/swagger](http://api.mytravels.local:8080/swagger) | — |
| Messaging health | [http://messaging.mytravels.local:8080/health](http://messaging.mytravels.local:8080/health) | — |
| MCP health | [http://mcp.mytravels.local:8080/health](http://mcp.mytravels.local:8080/health) | — |
| RabbitMQ Management | [http://rabbitmq.mytravels.local:8080](http://rabbitmq.mytravels.local:8080) | see `rabbitmq/1-secret.yaml` |
| MinIO Console | [http://minio.mytravels.local:8080](http://minio.mytravels.local:8080) | see `minio/1-secret.yaml` |
| Grafana | [http://grafana.mytravels.local:8080](http://grafana.mytravels.local:8080) | `user123` / `password123` |
| Prometheus | [http://prometheus.mytravels.local:8080](http://prometheus.mytravels.local:8080) | — |

**Non-browser endpoints:**

| Service | Endpoint | Used by |
|---|---|---|
| MCP (streamable HTTP) | `http://mcp.mytravels.local:8080/` | MCP clients — see `.mcp.json` and the port note in Step 11 |
| OTLP ingest | `http://otel.mytravels.local:8080` (HTTP) / `otel-collector:4317` (gRPC) | api, messaging, mcp, browser RUM |
| Tempo query API | `tempo:3200` (in-cluster only) | Grafana's Tempo datasource |
| postgres-exporter | `postgres-exporter:9187` (in-cluster only) | Prometheus |
| cadvisor | `cadvisor:8080` (in-cluster, headless) | Prometheus (DNS `SRV` discovery) |
| PostgreSQL | `127.0.0.1:5432` (TCP via Traefik) | psql, DBeaver, any SQL client |


---

## Step 17 — Diagnostics

Run these cells when a service is not behaving as expected.

In [ ]:
%%bash
echo "=== PostgreSQL logs ==="
kubectl logs -n mytravels-default -l app=postgres --tail=50

In [ ]:
%%bash
echo "=== RabbitMQ logs ==="
kubectl logs -n mytravels-default -l app=rabbitmq --tail=50

In [ ]:
%%bash
echo "=== MinIO logs ==="
kubectl logs -n mytravels-default -l app=minio --tail=50

In [39]:
%%bash
echo "=== API logs ==="
kubectl logs -n mytravels-default -l app=api --tail=50

=== API logs ===
info: Microsoft.EntityFrameworkCore.Database.Command[20101]
      Executed DbCommand (1ms) [Parameters=[@p0='?', @p1='?' (DbType = DateTime), @p2='?' (DbType = DateTime), @p3='?' (DbType = DateTime), @p4='?', @p5='?', @p6='?' (DbType = Boolean), @p7='?' (DbType = Double), @p8='?' (DbType = Double), @p9='?', @p10='?', @p11='?', @p12='?' (DbType = Guid)], CommandType='Text', CommandTimeout='30']
      INSERT INTO "PointOfInterests" ("Container", "DateCreated", "DateTaken", "DateUpdated", "FormattedAddress", "GeneratedBlobName", "ImageResized", "Latitude", "Longitude", "OriginalFileName", "PointOfInterestKey", "Reason", "UpdatedBy")
      VALUES (@p0, @p1, @p2, @p3, @p4, @p5, @p6, @p7, @p8, @p9, @p10, @p11, @p12)
      RETURNING "Id";
info: Microsoft.EntityFrameworkCore.Database.Command[20101]
      Executed DbCommand (10ms) [Parameters=[@p0='?', @p1='?' (DbType = DateTime), @p2='?' (DbType = DateTime), @p3='?' (DbType = DateTime), @p4='?', @p5='?', @p6='?' (DbType = Bool

In [ ]:
%%bash
echo "=== Messaging logs ==="
kubectl logs -n mytravels-default -l app=messaging --tail=50

In [ ]:
%%bash
echo "=== MCP logs ==="
kubectl logs -n mytravels-default -l app=mcp --tail=50

In [40]:
%%bash
echo "=== Web (nginx access/error) logs ==="
kubectl logs -n mytravels-default -l app=web --tail=50

=== Web (nginx access/error) logs ===
10.42.3.1 - - [23/Aug/2026:17:52:48 +0000] "GET / HTTP/1.1" 200 470 "-" "kube-probe/1.35" "-"
10.42.3.1 - - [23/Aug/2026:17:52:56 +0000] "GET / HTTP/1.1" 200 470 "-" "kube-probe/1.35" "-"
10.42.3.1 - - [23/Aug/2026:17:53:03 +0000] "GET / HTTP/1.1" 200 470 "-" "kube-probe/1.35" "-"
10.42.3.1 - - [23/Aug/2026:17:53:06 +0000] "GET / HTTP/1.1" 200 470 "-" "kube-probe/1.35" "-"
10.42.3.1 - - [23/Aug/2026:17:53:16 +0000] "GET / HTTP/1.1" 200 470 "-" "kube-probe/1.35" "-"
10.42.3.1 - - [23/Aug/2026:17:53:18 +0000] "GET / HTTP/1.1" 200 470 "-" "kube-probe/1.35" "-"
10.42.3.1 - - [23/Aug/2026:17:53:26 +0000] "GET / HTTP/1.1" 200 470 "-" "kube-probe/1.35" "-"
10.42.3.1 - - [23/Aug/2026:17:53:33 +0000] "GET / HTTP/1.1" 200 470 "-" "kube-probe/1.35" "-"
10.42.3.1 - - [23/Aug/2026:17:53:36 +0000] "GET / HTTP/1.1" 200 470 "-" "kube-probe/1.35" "-"
10.42.3.1 - - [23/Aug/2026:17:53:46 +0000] "GET / HTTP/1.1" 200 470 "-" "kube-probe/1.35" "-"
10.42.3.1 - - [23/Aug/

In [ ]:
%%bash
echo "=== Observability stack ==="
for APP in otel-collector prometheus tempo postgres-exporter cadvisor grafana; do
  echo "--- $APP ---"
  kubectl get pods -n mytravels-default -l app=$APP --no-headers 2>/dev/null || echo "(no pods)"
  kubectl logs -n mytravels-default -l app=$APP --tail=15 2>/dev/null | sed 's/^/    /'
  echo ""
done

In [ ]:
%%bash
# Recent events — useful for diagnosing scheduling or PVC binding failures
kubectl get events -n mytravels-default --sort-by='.lastTimestamp' | tail -20

---

## Step 18 — Seed Test Data (Optional)

The stack comes up with an empty database. This step bulk-loads a folder of real photos as points of interest so the Web app, the map, and the Grafana dashboards have something to show.

Each photo is POSTed to `POST /api/pointofinterest/image` as multipart form data, through the Traefik ingress at `http://api.mytravels.local:8080`. The API reads the GPS coordinates from the photo's EXIF metadata, stores the original in MinIO, and publishes to RabbitMQ — which is what drives the messaging worker's thumbnail resize and reverse-geocoding. So this also exercises the full async path end to end, not just the API: the resulting spans show up in Tempo under `service.name=mytravels-messaging`, and the queue depth moves on the RabbitMQ metrics Prometheus scrapes on 15692.

**Photos must be geotagged.** A file whose EXIF carries no GPS data is rejected with HTTP 403 and reported as a failure; the run continues past it to the next file.

Set `PHOTOS_DIR` in the cell below to a folder of photos — non-recursive, matching `.jpg`, `.jpeg`, `.png`, `.heic`, `.heif`. If that folder does not exist the cell skips itself, so this step stays safe when running the notebook top to bottom.

Uploads stream from disk with `curl`, so originals are sent whole — files of 6–16 MB are normal. Kestrel's default request-body limit is 30 MB, so anything larger will come back as HTTP 413.

The loop lives in `../.claude/scripts/upload-photos.sh`; it prints one TAB-separated record per file (name, `OK`/`FAIL`, id or reason) and a `TOTAL` line.

> **Dependencies:** `api.mytravels.local` must resolve (Step 3), the ingress must be applied (Step 15), and RabbitMQ, MinIO, and the messaging worker must be running for the thumbnails and addresses to fill in.
>
> Once the uploads finish the points show up at [http://web.mytravels.local:8080](http://web.mytravels.local:8080) — the deployed `web:v1.0.5` image was built with `VITE_API_BASE_URL=http://api.mytravels.local:8080` (see the header comment in `manifests/web/2-deployment.yaml`), so the browser reaches the API through the same ingress these uploads used.

In [36]:
%%bash
# Point PHOTOS_DIR at a folder of geotagged photos.
# The cell skips itself if the folder does not exist, so it is safe to run top-to-bottom.
PHOTOS_DIR="${PHOTOS_DIR:-$HOME/Personal/photos}"
API_BASE="http://api.mytravels.local:8080"

if [ ! -d "$PHOTOS_DIR" ]; then
  echo "SKIPPED — not a directory: $PHOTOS_DIR"
  echo "Set PHOTOS_DIR above to a folder of geotagged photos and re-run this cell to seed data."
  exit 0
fi

if ! curl -s -o /dev/null -m 5 "$API_BASE/api/pointofinterest"; then
  echo "API not reachable at $API_BASE — diagnosing inline:"
  echo "--- pods (app=api) ---"
  kubectl get pods -n mytravels-default -l app=api -o wide
  echo "--- ingress ---"
  kubectl get ingress api-ingress -n mytravels-default
  echo "--- api logs (last 30) ---"
  kubectl logs -n mytravels-default -l app=api --tail=30
  exit 1
fi

echo "=== Uploading photos from: $PHOTOS_DIR ==="
../.claude/scripts/upload-photos.sh "$PHOTOS_DIR" "$API_BASE"

=== Uploading photos from: /Users/tshepo.ntlhokoa/Personal/photos ===
IMG_20240125_183814.jpg	OK	1
IMG_20240125_184441.jpg	OK	2
IMG_20240125_184456.jpg	OK	3
IMG_20240125_192941.jpg	OK	4
IMG_20240323_095016.jpg	FAIL	HTTP 500: Image is not geocoded
IMG_20240323_095250.jpg	FAIL	HTTP 500: Image is not geocoded
IMG_20240323_184310~2.jpg	FAIL	HTTP 500: Image is not geocoded
IMG_20240525_110306.jpg	FAIL	HTTP 500: Image is not geocoded
IMG_20240629_224742.jpg	FAIL	HTTP 500: Image is not geocoded
IMG_20240706_150709.jpg	FAIL	HTTP 500: Image is not geocoded
IMG_20240706_151038.jpg	FAIL	HTTP 500: Image is not geocoded
IMG_20240706_151040.jpg	FAIL	HTTP 500: Image is not geocoded
IMG_20241215_130502.jpg	FAIL	HTTP 500: Image is not geocoded
IMG_20241215_130648.jpg	FAIL	HTTP 500: Image is not geocoded
IMG_20241215_131523.jpg	FAIL	HTTP 500: Image is not geocoded
IMG_20241215_131529.jpg	FAIL	HTTP 500: Image is not geocoded
IMG_20241224_111223.jpg	FAIL	HTTP 500: Image is not geocoded
IMG_20241224_112119

In [ ]:
%%bash
echo "=== Points of interest now in the database ==="
curl -s http://api.mytravels.local:8080/api/pointofinterest | python3 -c "
import json, sys
pois = json.load(sys.stdin)
print(f'{len(pois)} point(s) of interest')
for p in pois[:10]:
    print(' ', p.get('id'), '|', p.get('formattedAddress') or '(address pending geocoding)')
if len(pois) > 10:
    print(f'  ... and {len(pois) - 10} more')
" || {
  echo "API not reachable or returned non-JSON — diagnosing inline:"
  kubectl get pods -n mytravels-default -l app=api -o wide
  kubectl logs -n mytravels-default -l app=api --tail=50
}
echo ""
echo "Thumbnails and formatted addresses are filled in asynchronously by the messaging worker:"
kubectl logs -n mytravels-default -l app=messaging --tail=15
echo ""
echo "View the points on the map: http://web.mytravels.local:8080"

---

## Step 19 — Teardown

Delete all resources and the cluster. Run cells individually to tear down selectively, or run all to wipe everything.

In [45]:
%%bash
# Delete all manifests in reverse order
kubectl delete -f manifests/9-ingress.yaml
kubectl delete -f manifests/8-traefik-config.yaml
kubectl delete -f manifests/observability/
kubectl delete -f manifests/web/
kubectl delete -f manifests/mcp/
kubectl delete -f manifests/messaging/
kubectl delete -f manifests/api/
kubectl delete -f manifests/minio/
kubectl delete -f manifests/rabbitmq/
kubectl delete -f manifests/migrations/
kubectl delete -f manifests/postgres/
kubectl delete -f manifests/1-namespace.yaml

ingress.networking.k8s.io "management-ingress" deleted from mytravels-default namespace
ingress.networking.k8s.io "api-ingress" deleted from mytravels-default namespace
ingress.networking.k8s.io "messaging-ingress" deleted from mytravels-default namespace
ingress.networking.k8s.io "web-ingress" deleted from mytravels-default namespace
ingress.networking.k8s.io "mcp-ingress" deleted from mytravels-default namespace
ingressroutetcp.traefik.io "postgres-tcp-ingress" deleted from mytravels-default namespace
helmchartconfig.helm.cattle.io "traefik" deleted from kube-system namespace
configmap "otel-collector-config" deleted from mytravels-default namespace
deployment.apps "tempo" deleted from mytravels-default namespace
service "tempo" deleted from mytravels-default namespace
deployment.apps "postgres-exporter" deleted from mytravels-default namespace
service "postgres-exporter" deleted from mytravels-default namespace
daemonset.apps "cadvisor" deleted from mytravels-default namespace
servi

In [46]:
%%bash
# Delete the entire cluster — removes all Docker containers and volumes
k3d cluster delete mytravels

INFO[0000] Deleting cluster 'mytravels'                 
INFO[0010] Deleting cluster network 'k3d-mytravels'     
INFO[0010] Deleting 1 attached volumes...               
INFO[0010] Removing cluster details from default kubeconfig... 
INFO[0010] Removing standalone kubeconfig file (if there is one)... 
INFO[0010] Successfully deleted cluster mytravels!      
